# Entra Agent ID — OBO setup (Lab2)

End-to-end **on-behalf-of (OBO)** provisioning for the Lab1 sidecar chat UI:

- **Part 1** — MSAL.js client SPA (equivalent to `setup-obo-client-app.ps1`).
- **Part 2** — Blueprint API URI, `access_as_user` scope, SPA permissions, and admin consent Client SPA → Blueprint (`setup-obo-blueprint.ps1` steps 1–4 and 6; step 5 skipped).
- **Part 3** — Weather API delegated scope `weather.read` and agent → Weather admin consent (`setup-weather-api-resource.ps1` steps 3b and 6).

## Prerequisites

- Python 3.10+ and Jupyter (VS Code / Cursor).
- **Lab1** ids from [`entra-agent-id-setup.ipynb`](../../Lab1/scripts/entra-agent-id-setup.ipynb) or [`Lab1/sidecar/.env`](../../Lab1/sidecar/.env): `BLUEPRINT_APP_ID`, `AGENT_CLIENT_ID`, **`WEATHER_API_APP_ID`** (Weather app registration + `api://` URI).
- **Admin sign-in** uses **device code** (same public client as `Connect-MgGraph`).
- Graph **delegated** permissions (admin consent if required): `User.Read`, **`Application.ReadWrite.All`**, **`AgentIdentityBlueprint.ReadWrite.All`**, **`DelegatedPermissionGrant.ReadWrite.All`**.
- Install dependencies: run the pip cell below (`requirements.txt` in this folder).

## Security

No client secrets in this notebook. Avoid committing saved outputs with tenant-specific app IDs if your policy requires it.

## Graph API version

- **Part 1** and **Part 3** — Microsoft Graph **v1.0**
- **Part 2** — **beta** for blueprint application GET/PATCH


In [ ]:
%pip install -q -r requirements.txt


In [ ]:
import random
import time
import uuid

import httpx
import msal

TENANT_ID = "293882e2-ff46-4747-b46a-a84ca8e98fa0"  # @@@ YOUR TENANT ID @@@
BLUEPRINT_APP_ID = "0d0323fc-38a0-4e67-8ea1-fef157e367b3"  # @@@ blueprint appId from Lab1 @@@
AGENT_CLIENT_ID = "4cbccd31-3166-47fb-9e99-cfb4ac3b3595"  # @@@ agent appId from Lab1 @@@
WEATHER_API_APP_ID = "c3c5ffbf-98dd-429a-bf92-206f5c595610"  # @@@ weather API appId from Lab1 @@@

WEATHER_DELEGATED_SCOPE_VALUE = "weather.read"

RUN_SUFFIX = random.randint(100000, 999999)
APP_DISPLAY_NAME = f"Agent Demo Client SPA {RUN_SUFFIX}"
REDIRECT_URI = "http://localhost:3003"
CLIENT_SPA_APP_ID = None

GRAPH_ROOT = "https://graph.microsoft.com/v1.0"
GRAPH_BETA_ROOT = "https://graph.microsoft.com/beta"
GRAPH_POWERSHELL_APP_CLIENT_ID = "14d82eec-204b-4c2f-b7e8-296a70dab67e"

DELEGATED_SCOPES = [
    "https://graph.microsoft.com/User.Read",
    "https://graph.microsoft.com/Application.ReadWrite.All",
    "https://graph.microsoft.com/AgentIdentityBlueprint.ReadWrite.All",
    "https://graph.microsoft.com/DelegatedPermissionGrant.ReadWrite.All",
]

if "00000000" in TENANT_ID:
    raise ValueError("Set TENANT_ID in the configuration cell before continuing.")
print(f"Run suffix: {RUN_SUFFIX}")
print(f"SPA display name: {APP_DISPLAY_NAME}")
print(f"Redirect URI: {REDIRECT_URI}")
if BLUEPRINT_APP_ID:
    print(f"Blueprint app id (Part 2): {BLUEPRINT_APP_ID}")
else:
    print("BLUEPRINT_APP_ID not set yet — required before Part 2")
if AGENT_CLIENT_ID:
    print(f"Agent app id: {AGENT_CLIENT_ID}")
else:
    print("AGENT_CLIENT_ID not set yet — required before Part 2")
if WEATHER_API_APP_ID:
    print(f"Weather API app id (Part 3): {WEATHER_API_APP_ID}")
else:
    print("WEATHER_API_APP_ID not set yet — required before Part 3")


In [18]:
def acquire_delegated_token_device_code(
    tenant_id: str, scopes: list[str], client_id: str
) -> str:
    authority = f"https://login.microsoftonline.com/{tenant_id}"
    app = msal.PublicClientApplication(client_id, authority=authority)
    flow = app.initiate_device_flow(scopes=scopes)
    if "user_code" not in flow:
        raise RuntimeError(f"Device flow failed: {flow}")
    print(flow["message"])
    result = app.acquire_token_by_device_flow(flow)
    if "access_token" not in result:
        raise RuntimeError(f"Sign-in failed: {result.get('error_description', result)}")
    return result["access_token"]


def graph_request(
    method: str,
    url: str,
    token: str,
    json_body: dict | None = None,
) -> httpx.Response:
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
        "OData-Version": "4.0",
    }
    with httpx.Client(timeout=60.0) as client:
        response = client.request(method, url, headers=headers, json=json_body)
    if response.status_code >= 400:
        raise RuntimeError(f"{method} {url} -> {response.status_code}: {response.text}")
    return response


def normalize_oauth2_scope(scope: dict) -> dict:
    """Fields Graph expects on oauth2PermissionScopes PATCH (setup-weather-api-resource.ps1)."""
    return {
        "adminConsentDescription": str(scope.get("adminConsentDescription", "")),
        "adminConsentDisplayName": str(scope.get("adminConsentDisplayName", "")),
        "id": str(scope["id"]),
        "isEnabled": bool(scope.get("isEnabled", True)),
        "type": str(scope.get("type", "User")),
        "userConsentDescription": str(scope.get("userConsentDescription", "")),
        "userConsentDisplayName": str(scope.get("userConsentDisplayName", "")),
        "value": str(scope["value"]),
    }



## Admin sign-in (device code)

Run before **Part 1**. Re-run before **Part 2** or **Part 3** if you completed earlier parts with an older scope list (missing blueprint or delegated-permission grant scopes).


In [20]:
admin_token = acquire_delegated_token_device_code(
    TENANT_ID, DELEGATED_SCOPES, GRAPH_POWERSHELL_APP_CLIENT_ID
)
me = graph_request("GET", f"{GRAPH_ROOT}/me", admin_token).json()
print(f"Signed in as: {me.get('userPrincipalName', me.get('displayName'))}")


To sign in, use a web browser to open the page https://login.microsoft.com/device and enter the code H6L2YDRM8 to authenticate.
Signed in as: baoqger_gmail.com#EXT#@baoqgergmail.onmicrosoft.com


# Part 1 — Client SPA (MSAL.js)

Equivalent to `setup-obo-client-app.ps1`.


## Part 1 — Step 1 — Find or create SPA app registration


In [21]:
client_spa_app_id = CLIENT_SPA_APP_ID

if client_spa_app_id:
    lookup_url = (
        f"{GRAPH_ROOT}/applications?$filter=appId eq '{client_spa_app_id}'"
        "&$select=id,appId,displayName"
    )
    lookup = graph_request("GET", lookup_url, admin_token).json()
    if not lookup.get("value"):
        raise RuntimeError(f"No application for CLIENT_SPA_APP_ID={client_spa_app_id}")
    print(f"Using existing app: {lookup['value'][0]['displayName']} ({client_spa_app_id})")
else:
    safe_name = APP_DISPLAY_NAME.replace("'", "''")
    filter_url = (
        f"{GRAPH_ROOT}/applications?$filter=displayName eq '{safe_name}'"
        "&$select=id,appId,displayName"
    )
    existing = graph_request("GET", filter_url, admin_token).json()
    if existing.get("value"):
        client_spa_app_id = existing["value"][0]["appId"]
        print(f"Already exists: {client_spa_app_id}")
    else:
        create_body = {
            "displayName": APP_DISPLAY_NAME,
            "signInAudience": "AzureADMyOrg",
            "spa": {"redirectUris": [REDIRECT_URI]},
        }
        created = graph_request(
            "POST", f"{GRAPH_ROOT}/applications", admin_token, create_body
        ).json()
        client_spa_app_id = created["appId"]
        print(f"Created: {client_spa_app_id}")
        print("Waiting 5s for propagation...")
        time.sleep(5)

print(f"Client SPA App ID: {client_spa_app_id}")


Created: f219876d-98c6-4f10-a2da-9a3e0b8c0a89
Waiting 5s for propagation...
Client SPA App ID: f219876d-98c6-4f10-a2da-9a3e0b8c0a89


## Part 1 — Step 2 — Set SPA redirect URI


In [22]:
spa_body = {"spa": {"redirectUris": [REDIRECT_URI]}}
graph_request(
    "PATCH",
    f"{GRAPH_ROOT}/applications(appId='{client_spa_app_id}')",
    admin_token,
    spa_body,
)
print(f"Redirect URI set: {REDIRECT_URI}")


Redirect URI set: http://localhost:3003


## Part 1 — Step 3 — Ensure service principal


In [23]:
sp_url = (
    f"{GRAPH_ROOT}/servicePrincipals?$filter=appId eq '{client_spa_app_id}'&$select=id"
)
sp_result = graph_request("GET", sp_url, admin_token).json()
if sp_result.get("value"):
    print("Service principal already exists")
else:
    graph_request(
        "POST",
        f"{GRAPH_ROOT}/servicePrincipals",
        admin_token,
        {"appId": client_spa_app_id},
    )
    print("Created service principal")


Created service principal


## Part 1 — Done


In [24]:
print("Part 1 complete")
print(f"  CLIENT_SPA_APP_ID={client_spa_app_id}")
print(f"  Redirect URI: {REDIRECT_URI}")


Part 1 complete
  CLIENT_SPA_APP_ID=f219876d-98c6-4f10-a2da-9a3e0b8c0a89
  Redirect URI: http://localhost:3003


# Part 2 — Blueprint OBO setup

Equivalent to `setup-obo-blueprint.ps1` (steps 1–4 and 6). Requires `BLUEPRINT_APP_ID`, **`AGENT_CLIENT_ID`** (same as `-AgentAppId`; only step 5 in PowerShell), and `client_spa_app_id` from Part 1.


In [25]:
if not BLUEPRINT_APP_ID:
    raise ValueError("Set BLUEPRINT_APP_ID in the configuration cell before Part 2.")
if not AGENT_CLIENT_ID:
    raise ValueError("Set AGENT_CLIENT_ID in the configuration cell (Lab1 sidecar .env).")

print(f"Blueprint:  {BLUEPRINT_APP_ID}")
print(f"Agent:      {AGENT_CLIENT_ID}  (notebook does not grant Graph consent; PS step 5 only)")
print(f"Client SPA: {client_spa_app_id}")

agent_sp = graph_request(
    "GET",
    f"{GRAPH_ROOT}/servicePrincipals?$filter=appId eq '{AGENT_CLIENT_ID}'&$select=id,displayName",
    admin_token,
).json()
if not agent_sp.get("value"):
    raise RuntimeError(f"No service principal for AGENT_CLIENT_ID={AGENT_CLIENT_ID}")
print(f"Agent SP ok: {agent_sp['value'][0].get('displayName')}")


ValueError: Set AGENT_CLIENT_ID in the configuration cell (Lab1 sidecar .env).

## Part 2 — Step 1 — Look up blueprint


In [ ]:
lookup_url = (
    f"{GRAPH_BETA_ROOT}/applications?$filter=appId eq '{BLUEPRINT_APP_ID}'"
    "&$select=id,displayName,identifierUris,api"
)
result = graph_request("GET", lookup_url, admin_token).json()
if not result.get("value"):
    raise RuntimeError(f"No blueprint application for BLUEPRINT_APP_ID={BLUEPRINT_APP_ID}")

blueprint_app = result["value"][0]
blueprint_object_id = blueprint_app["id"]
existing_scopes = (blueprint_app.get("api") or {}).get("oauth2PermissionScopes") or []

print(f"Found: {blueprint_app.get('displayName')}")
print(f"Object ID: {blueprint_object_id}")
print(f"Current URIs: {', '.join(blueprint_app.get('identifierUris') or []) or '(none)'}")
scope_values = [s.get("value") for s in existing_scopes]
print(f"Current scopes: {', '.join(scope_values) or '(none)'}")


## Part 2 — Step 2 — Set Application ID URI


In [ ]:
app_id_uri = f"api://{BLUEPRINT_APP_ID}"
print(f"Application ID URI → {app_id_uri}")

identifier_uris = blueprint_app.get("identifierUris") or []
if app_id_uri in identifier_uris:
    print("Already set")
else:
    graph_request(
        "PATCH",
        f"{GRAPH_BETA_ROOT}/applications/{blueprint_object_id}",
        admin_token,
        {"identifierUris": [app_id_uri]},
    )
    print(f"Set App ID URI to {app_id_uri}")


## Part 2 — Step 3 — Add `access_as_user` delegated scope


In [ ]:
has_access_as_user = next(
    (s for s in existing_scopes if s.get("value") == "access_as_user"),
    None,
)

if has_access_as_user:
    access_as_user_scope_id = has_access_as_user["id"]
    print(f"Scope already exists (ID: {access_as_user_scope_id})")
else:
    access_as_user_scope_id = str(uuid.uuid4())
    scope_body = {
        "api": {
            "oauth2PermissionScopes": [
                {
                    "id": access_as_user_scope_id,
                    "adminConsentDescription": "Allow the application to access the agent on behalf of the signed-in user",
                    "adminConsentDisplayName": "Access agent as user",
                    "isEnabled": True,
                    "type": "User",
                    "userConsentDescription": "Allow the application to access the agent on your behalf",
                    "userConsentDisplayName": "Access agent as user",
                    "value": "access_as_user",
                }
            ]
        }
    }
    graph_request(
        "PATCH",
        f"{GRAPH_BETA_ROOT}/applications/{blueprint_object_id}",
        admin_token,
        scope_body,
    )
    print(f"Added access_as_user scope (ID: {access_as_user_scope_id})")
    print(f"Full scope: {app_id_uri}/access_as_user")


## Part 2 — Step 4 — API permission on Client SPA


In [ ]:
bp_updated = graph_request(
    "GET",
    f"{GRAPH_BETA_ROOT}/applications?$filter=appId eq '{BLUEPRINT_APP_ID}'&$select=id,api",
    admin_token,
).json()
scopes = (bp_updated["value"][0].get("api") or {}).get("oauth2PermissionScopes") or []
scope_id = next((s["id"] for s in scopes if s.get("value") == "access_as_user"), None)

spa_lookup = graph_request(
    "GET",
    f"{GRAPH_ROOT}/applications?$filter=appId eq '{client_spa_app_id}'&$select=id",
    admin_token,
).json()
client_spa_object_id = spa_lookup["value"][0]["id"] if spa_lookup.get("value") else None

if scope_id and client_spa_object_id:
    perm_body = {
        "requiredResourceAccess": [
            {
                "resourceAppId": BLUEPRINT_APP_ID,
                "resourceAccess": [{"id": scope_id, "type": "Scope"}],
            }
        ]
    }
    graph_request(
        "PATCH",
        f"{GRAPH_ROOT}/applications/{client_spa_object_id}",
        admin_token,
        perm_body,
    )
    print("Added API permission on Client SPA → Blueprint access_as_user")
else:
    print(f"Missing scope_id={scope_id} or client_spa_object_id={client_spa_object_id}")


## Skipped — Admin consent for Agent Identity → Graph (PS step 5)

In `setup-obo-blueprint.ps1`, **`-AgentAppId` is only used here** (oauth2 permission grant: agent → Microsoft Graph). Notebook steps 1–4 and 6 do not need the agent id on the wire; we still require `AGENT_CLIENT_ID` above so you can run the same parameters as the script.


In [ ]:
GRAPH_APP_ID = "00000003-0000-0000-c000-000000000000"
agent_sp_id = agent_sp["value"][0]["id"]
graph_sp = graph_request(
    "GET",
    f"{GRAPH_ROOT}/servicePrincipals?$filter=appId eq '{GRAPH_APP_ID}'&$select=id",
    admin_token,
).json()
graph_sp_id = graph_sp["value"][0]["id"] if graph_sp.get("value") else None

if graph_sp_id:
    grants_url = (
        f"{GRAPH_ROOT}/oauth2PermissionGrants?"
        f"$filter=clientId eq '{agent_sp_id}' and resourceId eq '{graph_sp_id}'"
    )
    existing = graph_request("GET", grants_url, admin_token).json()
    if existing.get("value"):
        print("Agent → Graph admin consent already present")
        print(f"  Scopes: {existing['value'][0].get('scope')}")
    else:
        print("Agent → Graph admin consent not found — run PowerShell step 5 if OBO needs it")

print()
print("PowerShell (full script or step 5 only):")
print(f"  pwsh ./setup-obo-blueprint.ps1 -TenantId {TENANT_ID} `")
print(f"    -BlueprintAppId {BLUEPRINT_APP_ID} `")
print(f"    -AgentAppId {AGENT_CLIENT_ID} `")
print(f"    -ClientSpaAppId {client_spa_app_id}")


## Part 2 — Step 6 — Admin consent for Client SPA → Blueprint


In [ ]:
blueprint_sp = graph_request(
    "GET",
    f"{GRAPH_ROOT}/servicePrincipals?$filter=appId eq '{BLUEPRINT_APP_ID}'&$select=id",
    admin_token,
).json()
blueprint_sp_id = blueprint_sp["value"][0]["id"] if blueprint_sp.get("value") else None

client_spa_sp = graph_request(
    "GET",
    f"{GRAPH_ROOT}/servicePrincipals?$filter=appId eq '{client_spa_app_id}'&$select=id",
    admin_token,
).json()
client_spa_sp_id = client_spa_sp["value"][0]["id"] if client_spa_sp.get("value") else None

if client_spa_sp_id and blueprint_sp_id:
    try:
        grants_url = (
            f"{GRAPH_ROOT}/oauth2PermissionGrants?"
            f"$filter=clientId eq '{client_spa_sp_id}' and resourceId eq '{blueprint_sp_id}'"
        )
        existing_consent = graph_request("GET", grants_url, admin_token).json()
        if existing_consent.get("value"):
            print("Admin consent already exists (Client SPA → Blueprint)")
        else:
            grant_body = {
                "clientId": client_spa_sp_id,
                "consentType": "AllPrincipals",
                "resourceId": blueprint_sp_id,
                "scope": "access_as_user",
            }
            graph_request(
                "POST",
                f"{GRAPH_ROOT}/oauth2PermissionGrants",
                admin_token,
                grant_body,
            )
            print("Admin consent granted for Client SPA → Blueprint")
    except RuntimeError as exc:
        print(f"Warning: {exc}")
else:
    print(
        f"Could not find service principals "
        f"(Client SPA: {client_spa_sp_id}, Blueprint: {blueprint_sp_id})"
    )


In [ ]:
print("OBO blueprint setup complete")
print(f"  Blueprint API:  {app_id_uri}")
print(f"  OBO scope:      {app_id_uri}/access_as_user")
print()
print(f"  MSAL.js scopes: ['{app_id_uri}/access_as_user']")
print(f"  User token (Tc) aud: {app_id_uri}")
print(f"  Sidecar AzureAd__ClientId: {BLUEPRINT_APP_ID}")


# Part 3 — Weather API OBO

Equivalent to [`setup-weather-api-resource.ps1`](../../Lab1/scripts/setup-weather-api-resource.ps1) **steps 3b and 6** only.

- **3b** — delegated scope `weather.read` on the Weather API app (OBO downstream audience).
- **6** — tenant-wide admin consent: Agent Identity → Weather API for that scope.

**Not in Part 3:** app role `Weather.Read`, agent app-role assignment, or `.env` updates (PS steps 3, 5, 7). Use the PowerShell script for autonomous (app-only) weather access.

**Prelude:** ensures the Weather API service principal exists (PS step 4) so step 6 can run.


In [ ]:
if not WEATHER_API_APP_ID:
    raise ValueError("Set WEATHER_API_APP_ID in the configuration cell before Part 3.")
if not AGENT_CLIENT_ID:
    raise ValueError("Set AGENT_CLIENT_ID in the configuration cell before Part 3.")
print(f"Weather API: {WEATHER_API_APP_ID}")
print(f"Agent:       {AGENT_CLIENT_ID}")
print(f"Scope:       {WEATHER_DELEGATED_SCOPE_VALUE}")


## Part 3 — Resolve Weather app and ensure service principal


In [ ]:
weather_lookup = graph_request(
    "GET",
    f"{GRAPH_ROOT}/applications?$filter=appId eq '{WEATHER_API_APP_ID}'"
    "&$select=id,appId,displayName,identifierUris",
    admin_token,
).json()
if not weather_lookup.get("value"):
    raise RuntimeError(f"No Weather application for WEATHER_API_APP_ID={WEATHER_API_APP_ID}")

weather_app = weather_lookup["value"][0]
weather_app_object_id = weather_app["id"]
weather_api_app_id = weather_app["appId"]
weather_app_id_uri = f"api://{weather_api_app_id}"

print(f"Found: {weather_app.get('displayName')}")
print(f"Object ID: {weather_app_object_id}")
print(f"App ID URI (expected): {weather_app_id_uri}")

if "agent_sp" not in globals() or not agent_sp.get("value"):
    agent_sp = graph_request(
        "GET",
        f"{GRAPH_ROOT}/servicePrincipals?$filter=appId eq '{AGENT_CLIENT_ID}'&$select=id,displayName",
        admin_token,
    ).json()
agent_sp_id = agent_sp["value"][0]["id"]
print(f"Agent SP: {agent_sp['value'][0].get('displayName')} ({agent_sp_id})")

weather_sp = graph_request(
    "GET",
    f"{GRAPH_ROOT}/servicePrincipals?$filter=appId eq '{weather_api_app_id}'&$select=id,appId",
    admin_token,
).json()
if weather_sp.get("value"):
    weather_sp_id = weather_sp["value"][0]["id"]
    print(f"Weather SP already exists: {weather_sp_id}")
else:
    created = graph_request(
        "POST",
        f"{GRAPH_ROOT}/servicePrincipals",
        admin_token,
        {"appId": weather_api_app_id},
    ).json()
    weather_sp_id = created["id"]
    print(f"Created Weather SP: {weather_sp_id}")
    print("Waiting 5s for propagation...")
    time.sleep(5)


## Part 3 — Step 3b — Delegated scope `weather.read` (OBO)


In [ ]:
api_snap = graph_request(
    "GET",
    f"{GRAPH_ROOT}/applications/{weather_app_object_id}?$select=api",
    admin_token,
).json()
api_block = api_snap.get("api") or {}
existing_scopes = [
    s for s in (api_block.get("oauth2PermissionScopes") or [])
    if s and s.get("value")
]

if any(s.get("value") == WEATHER_DELEGATED_SCOPE_VALUE for s in existing_scopes):
    print(f"Delegated scope already exists: {WEATHER_DELEGATED_SCOPE_VALUE}")
else:
    new_scope = {
        "id": str(uuid.uuid4()),
        "adminConsentDescription": "Allow the agent to read weather on behalf of the signed-in user",
        "adminConsentDisplayName": "Read weather as user",
        "isEnabled": True,
        "type": "User",
        "userConsentDescription": "Allow the agent to read weather on your behalf",
        "userConsentDisplayName": "Read weather as user",
        "value": WEATHER_DELEGATED_SCOPE_VALUE,
    }
    scope_by_value = {
        str(s["value"]): normalize_oauth2_scope(s) for s in existing_scopes
    }
    scope_by_value[WEATHER_DELEGATED_SCOPE_VALUE] = normalize_oauth2_scope(new_scope)
    token_version = int(api_block.get("requestedAccessTokenVersion") or 2)
    scope_body = {
        "api": {
            "requestedAccessTokenVersion": token_version,
            "oauth2PermissionScopes": list(scope_by_value.values()),
        }
    }
    try:
        graph_request(
            "PATCH",
            f"{GRAPH_ROOT}/applications/{weather_app_object_id}",
            admin_token,
            scope_body,
        )
        print(f"Added scope {weather_app_id_uri}/{WEATHER_DELEGATED_SCOPE_VALUE}")
        print("Waiting 5s for propagation...")
        time.sleep(5)
    except RuntimeError as exc:
        if "DuplicateValue" in str(exc):
            print("Delegated scope already present in tenant (DuplicateValue)")
        else:
            raise


## Part 3 — Step 6 — Admin consent Agent → Weather API (OBO)


In [ ]:
try:
    grant_filter = f"clientId eq '{agent_sp_id}' and resourceId eq '{weather_sp_id}'"
    grants_url = f"{GRAPH_ROOT}/oauth2PermissionGrants?$filter={grant_filter}"
    existing_grant = graph_request("GET", grants_url, admin_token).json()
    grant = (existing_grant.get("value") or [None])[0]

    if grant and WEATHER_DELEGATED_SCOPE_VALUE in (grant.get("scope") or "").split():
        print(f"Already consented (scope: {grant.get('scope')})")
    elif grant:
        merged = (grant.get("scope") or "").strip() + f" {WEATHER_DELEGATED_SCOPE_VALUE}"
        merged = " ".join(merged.split())
        graph_request(
            "PATCH",
            f"{GRAPH_ROOT}/oauth2PermissionGrants/{grant['id']}",
            admin_token,
            {"scope": merged},
        )
        print(f"Updated grant: {merged}")
    else:
        graph_request(
            "POST",
            f"{GRAPH_ROOT}/oauth2PermissionGrants",
            admin_token,
            {
                "clientId": agent_sp_id,
                "consentType": "AllPrincipals",
                "resourceId": weather_sp_id,
                "scope": WEATHER_DELEGATED_SCOPE_VALUE,
            },
        )
        print(f"Granted Agent → Weather API ({WEATHER_DELEGATED_SCOPE_VALUE})")
except RuntimeError as exc:
    print(f"Warning: OBO consent failed: {exc}")


In [ ]:
full_scope = f"{weather_app_id_uri}/{WEATHER_DELEGATED_SCOPE_VALUE}"
print("Weather OBO setup complete (Part 3)")
print(f"  Weather API:     {weather_app_id_uri}")
print(f"  Delegated scope: {full_scope}")
print(f"  Agent app id:    {AGENT_CLIENT_ID}")
print()
print("For autonomous Weather.Read (app role), run setup-weather-api-resource.ps1 steps 3 & 5.")


## After running

Copy **`CLIENT_SPA_APP_ID`** into [`Lab1/sidecar/.env`](../../Lab1/sidecar/.env) for the chat UI OBO flow.

Ensure **`WEATHER_API_APP_ID`** is already in `Lab1/sidecar/.env` from Lab1.

Optional: agent → Microsoft Graph consent — `setup-obo-blueprint.ps1` step 5; autonomous weather — `setup-weather-api-resource.ps1` steps 3 & 5.
